In [2]:
# %% [markdown]
# # Preparación del dataset
# 
# Este notebook:
# 1. Lee el dataset desde `data/processed/df.csv`
# 2. Elimina la variable `Nommodalidad`
# 3. Agrega los datos a nivel mensual (Anio + Mes o Fecha)
#    - Suma variables acumulativas
#    - Promedia variables de tasa
# 4. Muestra una vista previa del dataset final

# %%
import pandas as pd

# Configuración opcional para mejor visualización
pd.set_option('display.float_format', '{:,.3f}'.format)

# %% [markdown]
# ## 1. Carga de datos

# %%
path = "data/processed/df_master_estimacion_mensual_jalisco_aguacate_1996_2024.csv"
df = pd.read_csv(path, parse_dates=['Fecha'])

print("Shape original:", df.shape)
df.head()

# %% [markdown]
# ## 2. Definición de agregaciones
# 
# Se agrupa por Fecha (equivalente a Anio + Mes)
# 
# - Suma: variables acumulativas
# - Promedio: variables de tasa o proporción

# %%
# Variables a sumar
sum_cols = [
    'Sembrada',
    'Cosechada',
    'Siniestrada',
    'Volumenproduccion',
    'Valorproduccion'
]

# Variables a promediar
mean_cols = [
    'Rendimiento',
    'Precio',
    'share_mes'
]

# Otras columnas (tomar primer valor)
first_cols = [
    'Anio',
    'Mes',
    'Frecuencia',
    'TipoDato'
]

# %% [markdown]
# ## 3. Agregación

# %%
agg_dict = {col: 'sum' for col in sum_cols}
agg_dict.update({col: 'mean' for col in mean_cols})
agg_dict.update({col: 'first' for col in first_cols})

# Agrupar eliminando Nommodalidad implícitamente
df_agg = (
    df
    .groupby('Fecha', as_index=False)
    .agg(agg_dict)
    .sort_values('Fecha')
)

print("Shape agregado:", df_agg.shape)
df_agg.head()

# %% [markdown]
# ## 4. Reordenar columnas

# %%
cols_order = [
    'Fecha', 'Anio', 'Mes',
    'Sembrada', 'Cosechada', 'Siniestrada',
    'Volumenproduccion', 'Valorproduccion',
    'Rendimiento', 'Precio', 'share_mes',
    'Frecuencia', 'TipoDato'
]

# Filtrar solo columnas existentes (por seguridad)
cols_order = [c for c in cols_order if c in df_agg.columns]

df_agg = df_agg[cols_order]

df_agg.head()

# %% [markdown]
# ## 5. Vista para exportar como imagen
# 
# Esta salida es ideal para copiar/pegar como imagen en reportes.

# %%
display(df_agg.head(14))

# %% [markdown]
# ## 6. Validación: suma de share_mes por año
# 
# Se valida que la suma de `share_mes` por cada año sea aproximadamente 1.00

# %%
share_check = (
    df_agg
    .groupby('Anio')['share_mes']
    .sum()
    .reset_index()
)

# Redondeo para facilitar lectura
share_check['share_mes'] = share_check['share_mes'].round(6)

print("Validación de share_mes por año:")
display(share_check)

# Verificar si todos son ~1.0 (tolerancia numérica)
tolerance = 1e-6
invalid_years = share_check[abs(share_check['share_mes'] - 1.0) > tolerance]

if invalid_years.empty:
    print("✅ Todos los años suman aproximadamente 1.0")
else:
    print("⚠️ Años con inconsistencia en share_mes:")
    display(invalid_years)

# %% [markdown]
# ## 7. Guardar resultado (opcional)

# %%
out_path = "data/processed/df_aggregated.csv"
df_agg.to_csv(out_path, index=False)

print(f"Archivo guardado en: {out_path}")


Shape original: (696, 14)
Shape agregado: (348, 13)


,Fecha,Anio,Mes,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Valorproduccion,Rendimiento,Precio,share_mes,Frecuencia,TipoDato
0,1996-01-01,1996,1,"1,040.000",613.000,0.000,386.014,"639,625.551",0.543,"1,657.000",0.059,Mensual,Estimado
1,1996-02-01,1996,2,"1,040.000",613.000,0.000,479.086,"793,845.673",0.674,"1,657.000",0.073,Mensual,Estimado
2,1996-03-01,1996,3,"1,040.000",613.000,0.000,606.115,"1,004,332.765",0.853,"1,657.000",0.093,Mensual,Estimado
3,1996-04-01,1996,4,"1,040.000",613.000,0.000,636.277,"1,054,310.220",0.896,"1,657.000",0.097,Mensual,Estimado
4,1996-05-01,1996,5,"1,040.000",613.000,0.000,762.097,"1,262,795.553",1.073,"1,657.000",0.117,Mensual,Estimado
5,1996-06-01,1996,6,"1,040.000",613.000,0.000,504.284,"835,598.377",0.710,"1,657.000",0.077,Mensual,Estimado
6,1996-07-01,1996,7,"1,040.000",613.000,0.000,512.686,"849,519.977",0.722,"1,657.000",0.078,Mensual,Estimado
7,1996-08-01,1996,8,"1,040.000",613.000,0.000,522.846,"866,356.526",0.736,"1,657.000",0.080,Mensual,Estimado
8,1996-09-01,1996,9,"1,040.000",613.000,0.000,466.239,"772,558.083",0.656,"1,657.000",0.071,Mensual,Estimado
9,1996-10-01,1996,10,"1,040.000",613.000,0.000,589.228,"976,350.470",0.829,"1,657.000",0.090,Mensual,Estimado


Validación de share_mes por año:


,Anio,share_mes
0,1996,1.000
1,1997,1.000
2,1998,1.000
3,1999,1.000
4,2000,1.000
5,2001,1.000
6,2002,1.000
7,2003,1.000
8,2004,1.000
9,2005,1.000


✅ Todos los años suman aproximadamente 1.0
Archivo guardado en: data/processed/df_aggregated.csv
